# Exercise 11 — Autoformalization with Lean

*This notebook accompanies the guest lecture by **André Freitas** (Idiap Research Institute and University of Manchester), "Mathematical Reasoning and Autoformalization with Language Models". Parts 5 and 6 follow the [autoformalization tutorial](https://github.com/lanzhang128/multi_agent_autoformalization/blob/main/tutorial-demo-autoformalization.ipynb) of his group, built on MASA ([Zhang, Valentino and Freitas, 2025](https://aclanthology.org/2025.emnlp-demos.44/)).*

A language model can write a convincing proof that is wrong. A **proof assistant** such as [Lean](https://lean-lang.org) cannot be convinced: it accepts a proof only if every step follows from the rules of logic and from earlier results, which are collected in the [Mathlib](https://leanprover-community.github.io/mathlib4_docs/) library. **Autoformalization** is the task of translating informal mathematics (natural language and LaTeX) into such a formal language, so that a machine can check it.

There are two ways to fail: the formal code may **not compile** (the prover rejects it), or it may compile but **mean something else** than the original statement. You will meet both.

## What you will do
1. **Setup**: install Lean and Mathlib, and call Lean from Python.
2. **Lean in ten minutes**: theorems, goals and tactics.
3. **From informal to formal, by hand**: statements and proofs.
4. **Compiling is not enough**: formalizations that compile but change the meaning.
5. **A language model in the loop**: autoformalization with prover feedback.
6. **Is it faithful?** A language model as a judge.
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**2.1**, **2.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell. Run it and make sure it passes before you move on.

> **Compute and installation.** No GPU is needed for Lean. But Lean and Mathlib are a large installation: about **10 GB of disk** and **5–10 minutes** the first time. Checking a file then takes 10–30 seconds (the first check after starting can take a minute or two). We recommend **Google Colab** (a CPU runtime is enough with the Groq backend), where the installation does not touch your computer; it also works on Linux and macOS. **On Windows, use Colab** (or WSL). Colab forgets everything when the runtime stops, so you reinstall at the start of each session.
>
> Parts 5 and 6 use a language model, through the Groq API by default (roughly 20,000 tokens) or locally (then use a Colab GPU runtime, see Part 1).

In [ ]:
import hashlib
import json
import os
import re
import subprocess
import threading
from getpass import getpass
from pathlib import Path

## 1. Setup

### Installing Lean and Mathlib

The next cell installs [elan](https://github.com/leanprover/elan), Lean's version manager, and creates a Lean project that depends on Mathlib. `lake` is Lean's build tool; `lake exe cache get` downloads the compiled Mathlib library instead of compiling it yourself, which would take hours. If the cell is interrupted, simply run it again: finished steps are skipped.

In [ ]:
HOME = Path.home()
LEAN_PROJECT = HOME / "lean_autoformalization"  # the Lean project, with Mathlib (about 10 GB)
ELAN_BIN = HOME / ".elan" / "bin"
os.environ["PATH"] = f"{ELAN_BIN}{os.pathsep}{os.environ['PATH']}"


def run(command, cwd=None):
    print("$", command)
    result = subprocess.run(command, shell=True, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"command failed: {command}")


if not (ELAN_BIN / "lake").exists():
    run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o elan-init.sh"
        " && sh elan-init.sh -y --default-toolchain none")
if not (LEAN_PROJECT / "lakefile.toml").exists():
    # Creates a new project from Mathlib's template, with the Lean version that Mathlib currently uses
    run(f"lake +leanprover-community/mathlib4:lean-toolchain new {LEAN_PROJECT.name} math", cwd=LEAN_PROJECT.parent)
run("lake exe cache get", cwd=LEAN_PROJECT)
print("Lean and Mathlib are ready:", (LEAN_PROJECT / "lean-toolchain").read_text().strip())

### Calling Lean from Python

`check_lean(code)` writes the code to a file in the project, runs Lean on it, and returns a dictionary:
- `"ok"`: `True` if Lean found no errors,
- `"sorry"`: `True` if the code uses `sorry`, Lean's placeholder for a missing proof (more on that below),
- `"errors"`: the list of error messages,
- `"output"`: everything Lean printed.

If the code has no `import` line, `import Mathlib` is added at the top (so line numbers in the messages are 2 higher than in your code).

In [ ]:
def check_lean(code, timeout=900):
    if not re.search(r"^import ", code, flags=re.MULTILINE):
        code = "import Mathlib\n\n" + code
    (LEAN_PROJECT / "Scratch.lean").write_text(code, encoding="utf-8")
    result = subprocess.run(["lake", "env", "lean", "Scratch.lean"], cwd=LEAN_PROJECT,
                            capture_output=True, text=True, timeout=timeout)
    output = (result.stdout + result.stderr).strip()
    messages = re.split(r"\n(?=Scratch\.lean:\d+:\d+: )", output) if output else []
    errors = [m for m in messages if re.match(r"Scratch\.lean:\d+:\d+: error", m)]
    return {
        "ok": result.returncode == 0 and not errors,
        "sorry": bool(re.search(r"declaration uses .sorry.", output)),
        "errors": errors,
        "output": output,
    }


def show(result):
    print("✅ compiles" if result["ok"] else "❌ does not compile", "(uses sorry)" if result["sorry"] else "")
    if result["output"]:
        print(result["output"])


show(check_lean("#eval 2 + 2"))  # the first check loads Mathlib and can take a minute or two

## 2. Lean in ten minutes

In Lean, a **theorem** has a name, its **assumptions** (hypotheses) in parentheses, a **statement** after the colon, and a **proof** after `:=`:

```lean
theorem my_theorem (n : ℕ) (h : 0 < n) : n ≠ 0 := by
  omega
```

- `(n : ℕ)` says that `n` is a natural number. Every object has a **type**: `ℕ` (natural numbers, starting at 0), `ℤ` (integers), `ℝ` (real numbers), functions `ℝ → ℝ`, and so on.
- `(h : 0 < n)` is an assumption, with the name `h`, so that the proof can refer to it.
- `n ≠ 0` is the statement to prove, also called the **goal**.
- `by` starts a **tactic proof**: a sequence of commands that transform the goal until nothing is left to prove. `omega` is a tactic that solves linear arithmetic over ℕ and ℤ.
- `example` is a theorem without a name.

Some useful tactics:

| Tactic | What it does |
|--------|--------------|
| `norm_num` | proves numerical facts like `3 * 7 + 4 = 25` or `(2 : ℝ) < 3` |
| `omega` | linear arithmetic over ℕ and ℤ (`+`, `-`, `*` by constants, `<`, `=`) |
| `linarith` | linear arithmetic from the hypotheses, over ordered fields such as ℝ |
| `ring` | proves equalities that follow from the laws of commutative rings, like $(a+b)^2 = a^2 + 2ab + b^2$ |
| `positivity` | proves goals of the form `0 ≤ e` or `0 < e` when `e` is obviously positive (squares, sums of positives, …) |
| `simp` | simplifies the goal with many known rewriting rules |
| `exact h` | the goal is exactly `h` (an assumption or a theorem applied to arguments) |
| `sorry` | gives up: accepts the goal **without** proof (Lean warns about it) |

When a proof fails, Lean tells you what is left to prove. Try a false statement:

In [ ]:
show(check_lean("example : 2 + 2 = 5 := by norm_num"))

`norm_num` reduced the goal to `False`, which it cannot prove: `⊢ False` means "the goal left to prove is `False`". The prover's messages are precise; that will be useful when a language model needs feedback.

**2.1 Choose the tactics.** Replace every `sorry` with a tactic from the table so that the code compiles without `sorry`.

In [ ]:
LEAN_2_1 = """
example : 3 * 7 + 4 = 25 := by sorry
example (n : ℕ) : n < n + 1 := by sorry
example (x : ℝ) : 0 ≤ x ^ 2 + 1 := by sorry
example (a b : ℝ) : (a - b) * (a + b) = a ^ 2 - b ^ 2 := by sorry
example (x y : ℝ) (h1 : x < y) (h2 : y < 3) : x < 3 := by sorry
"""
show(check_lean(LEAN_2_1))

In [ ]:
# ✅ Check 2.1
result = check_lean(LEAN_2_1)
assert result["ok"], "the code does not compile: read the error messages above"
assert not result["sorry"], "replace every sorry with a tactic"
assert LEAN_2_1.count("example") == 5, "keep all five statements"
print("Looks good ✅")

## 3. From informal to formal, by hand

Take a statement and its informal proof:

> **Theorem.** The sum of two even natural numbers is even.
>
> **Proof.** Let $a$ and $b$ be even. Then $a = k + k$ and $b = m + m$ for some natural numbers $k$ and $m$. So $a + b = (k + m) + (k + m)$, which is even. $\blacksquare$

Formalizing it means making everything explicit: what kind of numbers, what "even" means, and every inference step. In Mathlib, `Even a` is *defined* as "there is an `r` with `a = r + r`". The formal proof follows the informal one step by step:

In [ ]:
EVEN_ADD = """
theorem even_add_even (a b : ℕ) (ha : Even a) (hb : Even b) : Even (a + b) := by
  obtain ⟨k, hk⟩ := ha      -- Let a be even: a = k + k for some k
  obtain ⟨m, hm⟩ := hb      -- and b = m + m for some m
  exact ⟨k + m, by omega⟩   -- a + b = (k + m) + (k + m), so it is even
"""
show(check_lean(EVEN_ADD))

The first line alone, with `sorry` as the proof, is already useful: it is the **formal statement**. Lean checks that it is well-formed (all names exist, all types fit) even without a proof. Translating only the statement is called *statement autoformalization*; translating the proof as well is *proof autoformalization*.

**3.1 Formalize a statement.** Write the theorem *"The square of an odd integer is odd"* in Lean, with `sorry` as the proof. Use `ℤ` for the integers and Mathlib's `Odd n`, which is defined as "there is a `k` with `n = 2 * k + 1`".

In [ ]:
ODD_SQUARE_STATEMENT = """
-- TODO
"""
show(check_lean(ODD_SQUARE_STATEMENT))

In [ ]:
# ✅ Check 3.1
result = check_lean(ODD_SQUARE_STATEMENT)
assert result["ok"], "the statement does not compile"
assert result["sorry"], "use sorry as the proof for now"
assert "ℤ" in ODD_SQUARE_STATEMENT and "Odd" in ODD_SQUARE_STATEMENT, "use ℤ and Odd"
print("Looks good ✅")

**3.2 Prove it.** Informally: if $n = 2k + 1$, then $n^2 = 4k^2 + 4k + 1 = 2(2k^2 + 2k) + 1$, which is odd. Replace `sorry` with a proof. *Hints*: `obtain ⟨k, rfl⟩ := h` takes the `k` out of `h` and replaces `n` by `2 * k + 1` everywhere. Then `exact ⟨2 * k ^ 2 + 2 * k, by ring⟩` provides the witness and lets `ring` check the algebra.

In [ ]:
ODD_SQUARE = """
-- TODO: your statement from 3.1, with a proof
"""
show(check_lean(ODD_SQUARE))

In [ ]:
# ✅ Check 3.2
result = check_lean(ODD_SQUARE)
assert result["ok"] and not result["sorry"], "the proof should compile without sorry"
print("Looks good ✅")

## 4. Compiling is not enough

Lean checks that a proof proves *the formal statement*. It cannot check that the formal statement means what the informal one says. Here are four formalizations of

> *If $n$ is an odd integer, then $n^2$ is odd.*

All four compile (with `sorry`), and all four are true.

In [ ]:
CANDIDATES = {
    "A": "theorem candidate (n : ℤ) (h : Odd n) : Odd (n ^ 2) := by sorry",
    "B": "theorem candidate (n : ℤ) (h : Even n) : Even (n ^ 2) := by sorry",
    "C": "theorem candidate (n : ℤ) (h1 : Odd n) (h2 : Even n) : Odd (n ^ 2) := by sorry",
    "D": "theorem candidate (n : ℤ) (h : Odd n) : 0 ≤ n ^ 2 := by sorry",
}
for name, code in CANDIDATES.items():
    result = check_lean(code)
    print(name, "compiles" if result["ok"] else "does not compile", "|", code)

**4.1 Which are faithful?** For each candidate, say whether it is a faithful formalization, and if not, what changed.

*Your answer here*

**4.2 A proof that proves nothing.** Prove candidate C. You do not need to think about squares at all: the assumptions contradict each other, and from a contradiction anything follows. *Hint*: take the witnesses out of `h1` and `h2` with `obtain`, as in Part 3, and `omega` will find the contradiction ($2a + 1 = b + b$ is impossible).

In [ ]:
VACUOUS = """
theorem candidate (n : ℤ) (h1 : Odd n) (h2 : Even n) : Odd (n ^ 2) := by
  sorry -- TODO
"""
show(check_lean(VACUOUS))

In [ ]:
# ✅ Check 4.2
result = check_lean(VACUOUS)
assert result["ok"] and not result["sorry"], "the proof should compile without sorry"
assert "(h1 : Odd n) (h2 : Even n) : Odd (n ^ 2)" in VACUOUS, "do not change the statement"
print("Looks good ✅")

**4.3 Types matter.** In mathematics, $(n - 1) + 1 = n$ for every number $n$. Lean's natural numbers stop at 0, so subtraction is *truncated*: `0 - 1 = 0`. Run the cell to see it. Then fix the statement below by adding the assumption that a mathematician would leave implicit (*"for every positive natural number n"*), and prove it with `omega`.

In [ ]:
show(check_lean("#eval (0 : ℕ) - 1\n#eval (0 : ℤ) - 1"))

In [ ]:
PRED_SUCC = """
theorem pred_succ (n : ℕ) : (n - 1) + 1 = n := by
  omega
"""
show(check_lean(PRED_SUCC))  # TODO: this fails for n = 0. Add the missing assumption.

In [ ]:
# ✅ Check 4.3
result = check_lean(PRED_SUCC)
assert result["ok"] and not result["sorry"], "the proof should compile without sorry"
assert "(n - 1) + 1 = n" in PRED_SUCC, "keep the conclusion; add an assumption instead"
print("Looks good ✅")

A model that writes `(n : ℕ)` where the text meant real numbers, or forgets an implicit assumption, produces code that compiles but states something else. This is why **semantic fidelity** is the hard part of evaluating autoformalization.

## 5. A language model in the loop

### Choosing a model

The notebook works with two **backends**; choose one in the next cell:

| `BACKEND` | Model | What you need |
|-----------|-------|---------------|
| `"groq"` (default) | `qwen/qwen3.8-27b` on the Groq API, as in weeks 7 and 10 | Your free Groq API key. No GPU. |
| `"local"` | `Qwen/Qwen2.5-1.5B-Instruct`, run on your machine with `transformers` | No account, but a GPU is strongly recommended: use a free **Colab** or **Kaggle** GPU runtime (in Colab: *Runtime → Change runtime type → T4 GPU*). On a laptop CPU every reply takes many seconds. |

The local model is about 18 times smaller than the one on Groq, so expect more mistakes. That is instructive too: compare your results with a classmate who uses the other backend.

`chat(messages)` sends a conversation (a list of `{"role": ..., "content": ...}` messages) to the model and returns the reply as a string. Replies are cached in a JSON file, so re-running a cell costs nothing. `usage` counts the calls and tokens, so you can compare what each pattern costs.

> **Keep your key secret.** Never write it into a code cell. If you set the environment variable `GROQ_API_KEY` before starting Jupyter, the cell does not ask at all.

In [ ]:
BACKEND = "groq"  # "groq" or "local"

if BACKEND == "groq":
    from groq import Groq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")
    client = Groq(max_retries=8)  # on a rate-limit error it waits and tries again (up to 8 times)
    # If this model is no longer available, pick another one from https://console.groq.com/docs/models
    MODEL = "qwen/qwen3.8-27b"
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device, "" if device == "cuda" else "(this will be slow; consider a Colab GPU runtime)")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    local_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto").to(device).eval()

CACHE_PATH = "llm_cache_autoformalization.json"
try:
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}
cache_lock = threading.Lock()
usage = {"calls": 0, "tokens": 0}


def _generate(messages, temperature, max_tokens, sample):
    # Returns the reply text and the number of tokens used
    if BACKEND == "groq":
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
        )
        return response.choices[0].message.content or "", response.usage.total_tokens
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    torch.manual_seed(sample)
    sampling = {"do_sample": True, "temperature": temperature} if temperature > 0 else {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
    with torch.no_grad():
        output = local_model.generate(**inputs, max_new_tokens=max_tokens, pad_token_id=tokenizer.eos_token_id, **sampling)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True), output.shape[1]


def chat(messages, temperature=0.0, max_tokens=512, sample=0):
    # Send a conversation to the model and return the reply (a string).
    # With temperature > 0, different values of `sample` give different (independently cached) samples.
    key = hashlib.sha256(json.dumps([BACKEND, MODEL, messages, temperature, max_tokens, sample]).encode()).hexdigest()
    if key not in cache:
        reply, tokens = _generate(messages, temperature, max_tokens, sample)
        with cache_lock:
            cache[key] = {"reply": reply, "tokens": tokens}
            with open(CACHE_PATH, "w", encoding="utf-8") as f:
                json.dump(cache, f)
    with cache_lock:
        usage["calls"] += 1
        usage["tokens"] += cache[key]["tokens"]
    return cache[key]["reply"]


print(chat([{"role": "user", "content": "Say hello to the NLP students in Aarhus in five words."}]))

### Zero-shot autoformalization

We ask the model to formalize statements only (with `sorry` as the proof), like the zero-shot agent in the MASA tutorial. Models usually wrap code in Markdown fences such as ```` ```lean ````, so we need to extract it.

**5.1 Implement `extract_lean(reply)`**: return the content of the first code block that starts with ```` ```lean ```` or ```` ```lean4 ```` (or just ```` ``` ````), stripped of surrounding whitespace. If there is no code block, return the whole reply, stripped. *Hint*: `re.search(r"```(?:lean4?)?\s*\n(.*?)```", reply, flags=re.DOTALL)`.

In [ ]:
def extract_lean(reply):
    ...  # TODO

In [ ]:
# ✅ Check extract_lean
assert extract_lean("Here it is:\n```lean\ntheorem t : 1 = 1 := by sorry\n```\nDone.") == "theorem t : 1 = 1 := by sorry"
assert extract_lean("```lean4\nimport Mathlib\nexample : True := by trivial\n```") == "import Mathlib\nexample : True := by trivial"
assert extract_lean("```\nexample : True := by trivial\n```") == "example : True := by trivial"
assert extract_lean("  example : True := by trivial ") == "example : True := by trivial"
print("Looks good ✅")

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in Lean 4 and Mathlib. You translate mathematical statements written in natural language "
    "into Lean 4 theorems (Lean 4, not Lean 3). Start the code with `import Mathlib`. Do not prove the theorems: use "
    "`sorry` as the proof. Only use names that exist in Mathlib. "
    "Reply with the Lean code in a single ```lean code block."
)

STATEMENTS = [
    "The sum of two even integers is even.",
    "For every real number x, x^2 - 2x + 1 is greater than or equal to 0.",
    "There are infinitely many prime numbers.",
    "Every differentiable function from the real numbers to the real numbers is continuous.",
    "Every natural number greater than 1 has a prime divisor.",
    "The square root of 2 is irrational.",
]


def first_attempt(statement):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Statement: {statement}"}]
    return messages, chat(messages, max_tokens=600)


messages, reply = first_attempt(STATEMENTS[0])
code = extract_lean(reply)
print(code)
show(check_lean(code))

### Refinement with prover feedback

When the code does not compile, Lean's error messages say exactly what is wrong and where. In the MASA tutorial, a *refinement agent* receives this feedback and revises the code. We do the same in a loop.

**5.2 Implement `formalize(statement, max_rounds=2)`.** Start with `first_attempt`, then check the code with Lean. As long as it does not compile and there are rounds left, continue the conversation: append the model's reply as an assistant message, and a user message with the first three error messages and the request to fix the code (in a single ```` ```lean ```` block). Return the last code and the list of `check_lean` results, one per attempt.

In [ ]:
def formalize(statement, max_rounds=2):
    messages, reply = first_attempt(statement)
    code = extract_lean(reply)
    results = [check_lean(code)]
    # TODO: the refinement loop
    return code, results

In [ ]:
formalizations = {}
for statement in STATEMENTS:
    code, results = formalize(statement)
    formalizations[statement] = code
    history = " → ".join("✅" if r["ok"] else "❌" for r in results)
    print(f"{history:10} {statement}\n{code}\n")
    if not results[-1]["ok"]:
        print("last errors:", results[-1]["errors"][:1], "\n")

In [ ]:
# ✅ Check formalize
code, results = formalize(STATEMENTS[0])
assert isinstance(code, str) and isinstance(results, list) and 1 <= len(results) <= 3
assert all(not r["ok"] for r in results[:-1]), "stop as soon as the code compiles"
print("Looks good ✅")

**5.3 Read the results.** How many statements compiled at the first attempt, and how many after refinement? What kinds of errors did you see (unknown names, type errors, syntax)? Pick one formalization that compiles and check by hand whether it is *faithful*.

*Your answer here*

## 6. Is it faithful? A language model as a judge

The prover is a **hard critic**: its verdict is certain, but only about well-formedness. For meaning, the MASA tutorial uses a **soft critic**: a language model that judges whether the formal statement captures the informal one. Let us test such a judge on the candidates of Part 4, where we know the right answers.

**6.1 Implement `judge(statement, lean_code)`.** Ask the model whether the Lean code faithfully formalizes the statement: same objects, same assumptions, same conclusion, nothing added or dropped. Ask for a short explanation, followed by a last line `Verdict: FAITHFUL` or `Verdict: UNFAITHFUL`. Return `True`, `False`, or `None` if there is no verdict.

In [ ]:
def judge(statement, lean_code):
    ...  # TODO

In [ ]:
STATEMENT = "If n is an odd integer, then n^2 is odd."
EXPECTED = {"A": True, "B": False, "C": False, "D": False}
correct = 0
for name, code in CANDIDATES.items():
    verdict = judge(STATEMENT, code)
    correct += verdict == EXPECTED[name]
    print(f"{name}: judge says {verdict}, expected {EXPECTED[name]}")
print(f"\nThe judge is right on {correct} of {len(CANDIDATES)} candidates")

In [ ]:
# ✅ Check judge
assert all(judge(STATEMENT, code) in (True, False, None) for code in CANDIDATES.values())
assert judge(STATEMENT, CANDIDATES["A"]) is not None, "the judge should give a verdict"
print("Looks good ✅")

In [ ]:
for statement, code in formalizations.items():
    print(f"{judge(statement, code)!s:6} {statement}")

**6.2 Hard and soft critics.** How reliable was the judge on the candidates? Why can we not simply trust an LLM judge for fidelity? Name one other way to check whether a formalization means what the statement says.

*Your answer here*

### Going further
- The [Natural Number Game](https://adam.math.hhu.de/#/g/leanprover-community/nng4) teaches Lean tactics as a puzzle game in the browser, and [Mathematics in Lean](https://leanprover-community.github.io/mathematics_in_lean/) is the standard tutorial. The [Lean web editor](https://live.lean-lang.org) runs Lean with Mathlib in your browser, without installation.
- The full [MASA tutorial](https://github.com/lanzhang128/multi_agent_autoformalization/blob/main/tutorial-demo-autoformalization.ipynb) adds retrieval of relevant Mathlib definitions, Isabelle as a second prover, and agents that combine all critics.
- [Autoformalization with Large Language Models](https://arxiv.org/abs/2205.12615) (Wu et al., 2022) started this line of work; [ProofNet](https://arxiv.org/abs/2302.12433) (Azerbayev et al., 2023) and [miniF2F](https://arxiv.org/abs/2109.00110) (Zheng et al., 2022) are standard benchmarks.

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. sorry

What does `sorry` do in a Lean proof?

A. It proves the goal by searching Mathlib<br>
B. It accepts the goal without proof; Lean compiles the file but warns that the declaration uses `sorry`<br>
C. It deletes the theorem<br>
D. It asks the user for a hint<br>

**Answer:** 

---

### 7.2. Compiling

A language model's formalization of a statement compiles in Lean (with `sorry` as the proof). What do you know?

A. The formal statement is well-formed: its names exist and its types fit<br>
B. The formal statement is true<br>
C. The formal statement means the same as the informal statement<br>
D. The informal statement is true<br>

**Answer:** 

---

### 7.3. Contradictory assumptions

A formal theorem has the assumptions `(h1 : x < 0) (h2 : 0 < x)`. Which is true?

A. Lean will refuse to compile it<br>
B. Any conclusion can be proved from these assumptions, so the theorem says nothing about any `x`<br>
C. The theorem can only be proved with `sorry`<br>
D. The theorem is false<br>

**Answer:** 

---

### 7.4. Prover feedback

Why is feedback from the prover particularly useful for refining a model's formalization?

A. It tells the model whether the formalization is faithful<br>
B. It is precise and reliable: it names the exact location and kind of error, such as an unknown identifier or a type mismatch<br>
C. It contains the correct formalization<br>
D. It makes the model's reply shorter<br>

**Answer:** 

---

### 7.5. Natural numbers

A model formalizes *"for all real numbers $x$, $x - 1 < x$"* as `∀ x : ℕ, x - 1 < x`. What is the problem?

A. There is none; ℕ and ℝ behave the same for subtraction<br>
B. The domain changed to natural numbers, and with truncated subtraction the statement becomes false for `x = 0`<br>
C. Lean does not support `∀`<br>
D. `<` is not defined on ℕ<br>

**Answer:** 